# 05 – SQL analysis: where does the money go?

Business questions answered with SQL on the cleaned payments table. Supplier rankings use the **Procurement** group only and exclude `REDACTED` (private individuals).

In [11]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/processed/procurement.db")

def q(sql):
    return pd.read_sql(sql, conn)

pd.options.display.float_format = "{:,.1f}".format

## 1. How did spend change between financial years?

In [12]:
q("""
SELECT CASE WHEN report_month < '2025-04' THEN '2024/25' ELSE '2025/26' END AS financial_year,
       spend_group,
       COUNT(*) AS payments,
       ROUND(SUM(amount) / 1e6, 1) AS total_millions
FROM payments
GROUP BY financial_year, spend_group
ORDER BY spend_group, financial_year
""")

,financial_year,spend_group,payments,total_millions
0,2024/25,Care and support,24092,209.5
1,2025/26,Care and support,25180,227.9
2,2024/25,Procurement,41755,400.3
3,2025/26,Procurement,39694,408.1
4,2024/25,Transfers and non-supplier,5042,193.2
5,2025/26,Transfers and non-supplier,6167,158.0


## 2. Who are the biggest suppliers (Procurement)?

In [13]:
q("""
SELECT supplier_clean,
       COUNT(*) AS payments,
       ROUND(SUM(amount) / 1e6, 2) AS total_millions,
       ROUND(100.0 * SUM(amount) / (
           SELECT SUM(amount) FROM payments
           WHERE spend_group = 'Procurement' AND supplier_clean <> 'REDACTED'
       ), 1) AS share_pct
FROM payments
WHERE spend_group = 'Procurement' AND supplier_clean <> 'REDACTED'
GROUP BY supplier_clean
ORDER BY total_millions DESC
LIMIT 15
""")

,supplier_clean,payments,total_millions,share_pct
0,STEPNELL LTD,230,38.5,4.8
1,COMENSURA LTD,219,35.2,4.4
2,AMERESCO LTD,1243,28.3,3.5
3,ETM CONTRACTORS LTD,730,26.1,3.3
4,CONSTELLIA PUBLIC LTD,1674,23.1,2.9
5,MEDEQUIP ASSISTIVE TECHNOLOGY LTD,60,20.5,2.6
6,ALUN GRIFFITHS CONTRACTORS LTD,306,16.9,2.1
7,WILLMOTT DIXON CONSTRUCTION LTD,48,15.9,2.0
8,ARCADIS LLP,690,15.6,1.9
9,JEFF WAY CONSTRUCTION LTD,5287,15.4,1.9


## 3. Are other public bodies hiding in Procurement?

In [14]:
q("""
SELECT supplier_clean,
       COUNT(*) AS payments,
       ROUND(SUM(amount) / 1e6, 2) AS total_millions
FROM payments
WHERE spend_group = 'Procurement'
  AND (supplier_clean LIKE '%COUNCIL%' OR supplier_clean LIKE '%NHS%'
       OR supplier_clean LIKE '%DEPARTMENT%' OR supplier_clean LIKE '%UNIVERSITY%'
       OR supplier_clean LIKE '% ICB%')
GROUP BY supplier_clean
ORDER BY total_millions DESC
LIMIT 15
""")

,supplier_clean,payments,total_millions
0,UNIVERSITY OF BRISTOL,15,0.3
1,UNIVERSITY OF EXETER,14,0.2
2,UNIVERSITY OF THE WEST OF ENGLAND,11,0.1
3,UNIVERSITY OF THE WEST OF ENGLAND TUITION FEES...,23,0.1
4,DEPARTMENT OF WORK AND PENSIONS,8,0.1
5,UNIVERSITY OF BRISTOL STUDENTS HEALTH SERVICE,10,0.1
6,UNIVERSITY OF BIRMINGHAM,11,0.0
7,DURHAM UNIVERSITY,1,0.0
8,OXFORD BROOKES UNIVERSITY,1,0.0
9,OXFORD UNIVERSITY PRESS,2,0.0


## 4. How concentrated is supplier spend?

In [15]:
q("""
WITH s AS (
    SELECT supplier_clean, SUM(amount) AS total
    FROM payments
    WHERE spend_group = 'Procurement' AND supplier_clean <> 'REDACTED'
    GROUP BY supplier_clean
    HAVING SUM(amount) > 0
),
r AS (
    SELECT total,
           ROW_NUMBER() OVER (ORDER BY total DESC) AS rnk,
           SUM(total) OVER (ORDER BY total DESC) AS running,
           SUM(total) OVER () AS grand
    FROM s
)
SELECT MAX(rnk) AS total_suppliers,
       MIN(CASE WHEN running / grand >= 0.5 THEN rnk END) AS suppliers_for_50pct,
       MIN(CASE WHEN running / grand >= 0.8 THEN rnk END) AS suppliers_for_80pct
FROM r
""")

,total_suppliers,suppliers_for_50pct,suppliers_for_80pct
0,3345,30,152


## Findings so far

All figures are net of credits and exclude `REDACTED` payments to individuals.

1. **Procurement spend grew modestly:** £400.3m in 2024/25 to £408.1m in 2025/26 (+1.9%), while the number of payments fell 4.9% (41,755 to 39,694). The average payment rose from about £9,600 to £10,300.
2. **Care and support grew faster:** £209.5m to £227.9m (+8.8%).
3. **Transfers and non-supplier payments fell 18%:** £193.2m to £158.0m.
4. **No single supplier dominates.** The largest, Stepnell Ltd, is 4.8% of Procurement spend, and the top 15 together are about 36%.
5. **But a small group carries most of the value:** 30 of 3,345 suppliers (0.9%) account for half of Procurement spend, and 152 (4.5%) account for 80%. Review effort for value should focus on these.
6. **Two suppliers have unusual payment profiles to test next:** Jeff Way Construction (5,287 payments averaging about £2,900) and Medequip Assistive Technology (60 payments averaging about £340,000).